# 03 — Data cleaning: FAERS GLP-1 reports
**Owner:** Brian  ·  **Output:** Logistic Regression Model Based on EDA

| Step | Details            |
|------|--------------------|
| 0    | Setup              |
| 1    | Load the Data      |
| 2    | Data Split (80:20) |
| 3    | Preprocessing      |
| 4    | Model              |
| 5    | Evaluation         |

## Step 0 -- Setup

In [71]:
# Libraries
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report, average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression

## Step 1 -- Load the Data

In [72]:
# Load the data
df = pd.read_parquet('../data/03_model/glp1_model_ready.parquet')
df.head()

,primaryid,caseid,drug,age_group,sex,reporter_type,indication_group,route_group,dose_reported,year,...,rg_injection_site,rg_lack_of_effect,rg_liver,rg_mental_health_sleep,rg_muscle_bone_injury,rg_nervous_system,rg_other,rg_pancreas_gallbladder,rg_skin_hair,rg_weight_appetite
0,1300919515,13009195,liraglutide,40-64,M,Physician,Diabetes,Injection,0,2022,...,0,0,0,1,1,1,1,1,1,1
1,138550922,13855092,dulaglutide,Unknown,M,Consumer,Unknown,Injection,1,2022,...,1,0,0,0,0,0,0,0,0,0
2,145908423,14590842,liraglutide,40-64,F,Consumer,Unknown,Injection,0,2022,...,1,0,0,0,1,0,1,0,0,1
3,159832903,15983290,dulaglutide,Unknown,M,Consumer,Other,Injection,1,2022,...,1,0,0,0,0,1,0,0,0,0
4,164724373,16472437,dulaglutide,Unknown,M,Consumer,Diabetes,Unknown,1,2022,...,0,1,0,0,0,0,0,0,0,0


In [73]:
# Drop ID columns
dropped_columns = [
    'primaryid',
    'caseid'
]
df = df.drop(columns=dropped_columns)
df.columns

Index(['drug', 'age_group', 'sex', 'reporter_type', 'indication_group',
       'route_group', 'dose_reported', 'year', 'n_reactions', 'n_other_drugs',
       'is_serious', 'rg_allergy', 'rg_blood_sugar_fluids_kidney',
       'rg_cancer_thyroid', 'rg_dosing_device_product_use', 'rg_eye',
       'rg_gastrointestinal', 'rg_general_tiredness_pain',
       'rg_heart_circulation', 'rg_infections_breathing', 'rg_injection_site',
       'rg_lack_of_effect', 'rg_liver', 'rg_mental_health_sleep',
       'rg_muscle_bone_injury', 'rg_nervous_system', 'rg_other',
       'rg_pancreas_gallbladder', 'rg_skin_hair', 'rg_weight_appetite'],
      dtype='str')

In [74]:
# Data type conversion (Based on decision table)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 252568 entries, 0 to 252567
Data columns (total 30 columns):
 #   Column                        Non-Null Count   Dtype
---  ------                        --------------   -----
 0   drug                          252568 non-null  str  
 1   age_group                     252568 non-null  str  
 2   sex                           252568 non-null  str  
 3   reporter_type                 252568 non-null  str  
 4   indication_group              252568 non-null  str  
 5   route_group                   252568 non-null  str  
 6   dose_reported                 252568 non-null  int64
 7   year                          252568 non-null  int64
 8   n_reactions                   252568 non-null  int64
 9   n_other_drugs                 252568 non-null  int64
 10  is_serious                    252568 non-null  int64
 11  rg_allergy                    252568 non-null  int64
 12  rg_blood_sugar_fluids_kidney  252568 non-null  int64
 13  rg_cancer_thyroid        

In [ ]:
# Define categorical and numeric columns
cat_cols = [
    'drug',
    'age_group',
    'sex',
    'reporter_type',
    'indication_group',
    'route_group'
]
num_cols = df.columns.difference(cat_cols).tolist() # All other categories are numeric
num_cols.remove('is_serious')
df[cat_cols] = df[cat_cols].astype('category')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 252568 entries, 0 to 252567
Data columns (total 30 columns):
 #   Column                        Non-Null Count   Dtype   
---  ------                        --------------   -----   
 0   drug                          252568 non-null  category
 1   age_group                     252568 non-null  category
 2   sex                           252568 non-null  category
 3   reporter_type                 252568 non-null  category
 4   indication_group              252568 non-null  category
 5   route_group                   252568 non-null  category
 6   dose_reported                 252568 non-null  int64   
 7   year                          252568 non-null  int64   
 8   n_reactions                   252568 non-null  int64   
 9   n_other_drugs                 252568 non-null  int64   
 10  is_serious                    252568 non-null  int64   
 11  rg_allergy                    252568 non-null  int64   
 12  rg_blood_sugar_fluids_kidney  252568 non-

## Step 2 -- Data Split (80:20)

In [ ]:
# Split the data into training and test sets
y = df['is_serious']
X = df.drop(columns=['is_serious'])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

Training set size: 202054
Test set size: 50514


## Step 3 -- Preprocessing

In [ ]:
# Preprocessing pipelines for numeric and categorical data
num_pl = Pipeline([
    ('scaler', RobustScaler())
])
cat_pl = Pipeline([
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])
preprocessor = ColumnTransformer([
    ('num', num_pl, num_cols),
    ('cat', cat_pl, cat_cols)
])

## Step 4 -- Model

In [ ]:
# Build the full pipeline with preprocessing and classifier
full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
])
full_pipeline.fit(X_train, y_train)
predictions = full_pipeline.predict(X_test)

## Step 5 -- Evaluate

In [ ]:
# Scores
print(f'Classification Report:\n{classification_report(y_test, predictions)}')
print(f'Confusion Matrix:\n{confusion_matrix(y_test, predictions)}')
print(f'Average Precision Score: {average_precision_score(y_test, predictions)}')
print(f'ROC AUC Score: {roc_auc_score(y_test, predictions)}')

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.78      0.86     44719
           1       0.31      0.78      0.44      5795

    accuracy                           0.78     50514
   macro avg       0.64      0.78      0.65     50514
weighted avg       0.89      0.78      0.81     50514

Confusion Matrix:
[[34770  9949]
 [ 1299  4496]]
Average Precision Score: 0.2671958942130839
ROC AUC Score: 0.7766815505840797
